In [23]:
import pdfplumber
from pathlib import Path

CAMINHO_PDF = Path("../data/raw/anuario_cerveja_2026.pdf")

with pdfplumber.open(CAMINHO_PDF) as pdf:
    print("Total de páginas:", len(pdf.pages))

    for numero_tabela in range(16, 24):
        rotulo = f"Tabela {numero_tabela}:"
        paginas_encontradas = []

        for indice, pagina in enumerate(pdf.pages):
            texto = pagina.extract_text() or ""
            if rotulo in texto:
                paginas_encontradas.append(indice)

        print(rotulo, "índices", paginas_encontradas)

Total de páginas: 64
Tabela 16: índices [47]
Tabela 17: índices [49]
Tabela 18: índices [50]
Tabela 19: índices [51]
Tabela 20: índices [51]
Tabela 21: índices [54]
Tabela 22: índices [55]
Tabela 23: índices [55]


In [24]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    pagina = pdf.pages[51]

    print("===== TEXTO =====")
    print(pagina.extract_text())

    tabelas = pagina.extract_tables()
    print("===== TABELAS DETECTADAS:", len(tabelas), "=====")

    for numero, tabela in enumerate(tabelas, start=1):
        print("--- tabela", numero, "---")
        for linha in tabela:
            print(linha)

===== TEXTO =====
A região nordeste, embora ocupe a terceira posição em números de cervejarias registradas,
com 130 estabelecimentos, ocupa a segunda posição quanto ao volume de produção
declarado, com 3.802.222.465,12 litros, o que representa 24,2% do volume nacional.
A região Sul, por sua vez, embora ocupando a segunda colocação quanto ao número de
cervejarias registradas, com 759 estabelecimentos, ocupa a terceira posição quanto ao
volume de produção declarado, com 2.447.510.384,25, o que representa 15,6% do volume
nacional.
Tal contradição deve-se à produção média verificada em cada região, que se encontram
em extremos opostos, sendo a da região Nordeste a maior, com 29.247.865,12 litros por
cervejaria, enquanto aquela da região sul é a menor, com 3.224.651,36 litros de cerveja
por estabelecimento.
Tabela 19: Análise da concentração do volume produzido de cerveja no Brasil
Porcentagem de Cervejarias Porcentagem relativa ao
Volume produzido
declarantes volume total produzido
1% 6.66

In [25]:
import pandas as pd

def converter_numero_br(texto):
    limpo = texto.replace("%", "").replace(".", "").replace(",", ".").strip()
    return float(limpo)

print(converter_numero_br("6.666.193.835,38"))
print(converter_numero_br("42,49%"))
print(converter_numero_br("-3,90"))

6666193835.38
42.49
-3.9


In [26]:
pd.options.display.float_format = "{:,.2f}".format

In [27]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    tabelas = pdf.pages[51].extract_tables()

linhas = tabelas[0][1:]

df_19 = pd.DataFrame(linhas, columns=["pct_cervejarias", "volume_litros", "pct_volume"])

for coluna in df_19.columns:
    df_19[coluna] = df_19[coluna].map(converter_numero_br)

df_19["ano_referencia"] = 2025
df_19["fonte"] = "Anuário da Cerveja 2026, Tabela 19, p. 52"
df_19["origem"] = "tabela"

TOTAL_2025 = 15_688_083_191.69

ponto_10 = pd.DataFrame([{
    "pct_cervejarias": 10.0,
    "volume_litros": round(TOTAL_2025 * 0.995, 2),
    "pct_volume": 99.5,
    "ano_referencia": 2025,
    "fonte": "Anuário da Cerveja 2026, texto da p. 52",
    "origem": "texto; volume derivado (99,5% do total)",
}])

df_19 = pd.concat([df_19, ponto_10], ignore_index=True)

PASTA_SAIDA = Path("../data/processed")
df_19.to_csv(PASTA_SAIDA / "tabela_19_concentracao_2025.csv", index=False)

df_19

,pct_cervejarias,volume_litros,pct_volume,ano_referencia,fonte,origem
0,1.00,"6,666,193,835.38",42.49,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52",tabela
1,2.00,"11,238,768,122.78",71.64,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52",tabela
2,3.00,"13,973,317,068.01",89.07,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52",tabela
3,4.00,"15,150,315,200.57",96.57,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52",tabela
4,5.00,"15,471,743,939.70",98.62,2025,"Anuário da Cerveja 2026, Tabela 19, p. 52",tabela
5,10.00,"15,609,642,775.73",99.50,2025,"Anuário da Cerveja 2026, texto da p. 52","texto; volume derivado (99,5% do total)"


In [28]:
linhas = tabelas[1][1:]

df_20 = pd.DataFrame(linhas, columns=["categoria", "volume_litros", "pct_volume", "variacao_pct"])

df_20["categoria"] = df_20["categoria"].str.replace("\n", " ")

for coluna in ["volume_litros", "pct_volume", "variacao_pct"]:
    df_20[coluna] = df_20[coluna].map(converter_numero_br)

df_20["ano_referencia"] = 2025
df_20["fonte"] = "Anuário da Cerveja 2026, Tabela 20, p. 52"

df_20.to_csv(PASTA_SAIDA / "tabela_20_puro_malte_2025.csv", index=False)

print("Soma dos volumes:", df_20["volume_litros"].sum())
df_20

Soma dos volumes: 15688083191.69


,categoria,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte
0,Cerveja Puro Malte ou 100% malte,"4,580,310,971.90",29.20,21.01,2025,"Anuário da Cerveja 2026, Tabela 20, p. 52"
1,Demais cervejas,"11,107,772,219.79",70.80,-3.90,2025,"Anuário da Cerveja 2026, Tabela 20, p. 52"


In [29]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    for indice in [49, 50]:
        tabelas = pdf.pages[indice].extract_tables()
        print("===== PÁGINA ÍNDICE", indice, "- tabelas detectadas:", len(tabelas), "=====")

        for posicao, tabela in enumerate(tabelas):
            print("--- posição", posicao, "---")
            for linha in tabela:
                print(linha)

===== PÁGINA ÍNDICE 49 - tabelas detectadas: 2 =====
--- posição 0 ---
['']
['']
--- posição 1 ---
['Região', 'Volume (L)', 'Proporção relativa ao\nvolume nacional (%)', 'Variação 2024/2025 (%)']
['Centro-Oeste', '1.054.248.799,25', '6,7', '-5,59']
['Nordeste', '3.802.222.465,12', '24,2', '-8,07']
['Norte', '215.667.721,98', '1,4', '-16,92']
['Sudeste', '8.168.433.821,09', '52,1', '-9,36']
['Sul', '2.447.510.384,25', '15,6', '-8,90']
===== PÁGINA ÍNDICE 50 - tabelas detectadas: 1 =====
--- posição 0 ---
['Região', 'Volume (L)', 'Nº de cervejarias', 'Produção Média (L/\ncervejaria)']
['Nordeste', '3.802.222.465,12', '130', '29.247.865,12']
['Centro-Oeste', '1.054.248.799,25', '98', '10.757.640,81']
['Norte', '215.667.721,98', '44', '4.901.539,14']
['Sul', '2.447.510.384,25', '759', '3.224.651,36']
['Sudeste', '8.168.433.821,09', '923', '8.849.874,13']


In [30]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    tabela_17 = pdf.pages[49].extract_tables()[1]
    tabela_18 = pdf.pages[50].extract_tables()[0]

df_17 = pd.DataFrame(tabela_17[1:], columns=["regiao", "volume_litros", "pct_volume", "variacao_pct"])
for coluna in ["volume_litros", "pct_volume", "variacao_pct"]:
    df_17[coluna] = df_17[coluna].map(converter_numero_br)
df_17["ano_referencia"] = 2025
df_17["fonte"] = "Anuário da Cerveja 2026, Tabela 17, p. 50"
df_17.to_csv(PASTA_SAIDA / "tabela_17_producao_regiao_2025.csv", index=False)

df_18 = pd.DataFrame(tabela_18[1:], columns=["regiao", "volume_litros", "n_cervejarias", "producao_media_litros"])
for coluna in ["volume_litros", "n_cervejarias", "producao_media_litros"]:
    df_18[coluna] = df_18[coluna].map(converter_numero_br)
df_18["n_cervejarias"] = df_18["n_cervejarias"].astype(int)
df_18["ano_referencia"] = 2025
df_18["fonte"] = "Anuário da Cerveja 2026, Tabela 18, p. 51"
df_18.to_csv(PASTA_SAIDA / "tabela_18_producao_media_regiao_2025.csv", index=False)

print("Soma das regiões (T17):", df_17["volume_litros"].sum())
print("Soma das regiões (T18):", df_18["volume_litros"].sum())
print("Total de cervejarias:", df_18["n_cervejarias"].sum())

display(df_17)
display(df_18)

Soma das regiões (T17): 15688083191.689999
Soma das regiões (T18): 15688083191.689999
Total de cervejarias: 1954


,regiao,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte
0,Centro-Oeste,"1,054,248,799.25",6.70,-5.59,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
1,Nordeste,"3,802,222,465.12",24.20,-8.07,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
2,Norte,"215,667,721.98",1.40,-16.92,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
3,Sudeste,"8,168,433,821.09",52.10,-9.36,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"
4,Sul,"2,447,510,384.25",15.60,-8.90,2025,"Anuário da Cerveja 2026, Tabela 17, p. 50"


,regiao,volume_litros,n_cervejarias,producao_media_litros,ano_referencia,fonte
0,Nordeste,"3,802,222,465.12",130,"29,247,865.12",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
1,Centro-Oeste,"1,054,248,799.25",98,"10,757,640.81",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
2,Norte,"215,667,721.98",44,"4,901,539.14",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
3,Sul,"2,447,510,384.25",759,"3,224,651.36",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"
4,Sudeste,"8,168,433,821.09",923,"8,849,874.13",2025,"Anuário da Cerveja 2026, Tabela 18, p. 51"


In [31]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    for indice in [47, 54, 55]:
        tabelas = pdf.pages[indice].extract_tables()
        print("===== PÁGINA ÍNDICE", indice, "- tabelas detectadas:", len(tabelas), "=====")

        for posicao, tabela in enumerate(tabelas):
            print("--- posição", posicao, "---")
            for linha in tabela:
                print(linha)

===== PÁGINA ÍNDICE 47 - tabelas detectadas: 1 =====
--- posição 0 ---
['UF', 'Estoque de empregos', 'Porcentagem Relativa', 'Variação Relativa']
['RO', '45', '0,11', '0,00%']
['AC', '11', '0,03', '0,00%']
['AM', '531', '1,27', '-2,57%']
['RR', '13', '0,03', '0,00%']
['PA', '883', '2,10', '-4,13%']
['AP', '5', '0,01', 'de 0 para 5']
['TO', '14', '0,03', '-30%']
['MA', '694', '1,65', '-3,74%']
['PI', '448', '1,07', '-0,67%']
['CE', '648', '1,54', '-14,29%']
['RN', '45', '0,11', '-21,05%']
['PB', '59', '0,14', '-3,51%']
['PE', '2.288', '5,45', '-3,74%']
['AL', '40', '0,10', '8,11%']
['SE', '417', '0,99', '-5,44%']
['BA', '2.234', '5,32', '-0,18%']
['MG', '4.720', '11,24', '3,74%']
['ES', '240', '0,57', '18,91%']
['RJ', '5.216', '12,43', '-3,19%']
['SP', '14.182', '33,79', '-2,82%']
['PR', '2.680', '6,38', '-5,50%']
['SC', '1.552', '3,70', '-8,81%']
['RS', '2.479', '5,91', '-1,70%']
['MS', '35', '0,08', '-7,89%']
['MT', '906', '2,16', '-3,10%']
['GO', '1.535', '3,66', '-1,85%']
['DF', '56

In [32]:
def converter_ou_nulo(texto):
    try:
        return converter_numero_br(texto)
    except ValueError:
        return float("nan")

with pdfplumber.open(CAMINHO_PDF) as pdf:
    tabela_16 = pdf.pages[47].extract_tables()[0]

df_16 = pd.DataFrame(tabela_16[1:], columns=["uf", "estoque_empregos", "pct_empregos", "variacao_texto"])

df_16["estoque_empregos"] = df_16["estoque_empregos"].map(converter_numero_br).astype(int)
df_16["pct_empregos"] = df_16["pct_empregos"].map(converter_numero_br)
df_16["variacao_pct"] = df_16["variacao_texto"].map(converter_ou_nulo)

df_16["ano_referencia"] = 2025
df_16["fonte"] = "Anuário da Cerveja 2026, Tabela 16, p. 48"
df_16.to_csv(PASTA_SAIDA / "tabela_16_empregos_uf_2025.csv", index=False)

print("UFs:", len(df_16))
print("Total de empregos:", df_16["estoque_empregos"].sum())
print("Soma das porcentagens:", df_16["pct_empregos"].sum())
display(df_16[df_16["variacao_pct"].isna()])

UFs: 27
Total de empregos: 41976
Soma das porcentagens: 100.0


,uf,estoque_empregos,pct_empregos,variacao_texto,variacao_pct,ano_referencia,fonte
5,AP,5,0.01,de 0 para 5,NaN,2025,"Anuário da Cerveja 2026, Tabela 16, p. 48"


In [33]:
with pdfplumber.open(CAMINHO_PDF) as pdf:
    tabela_21 = pdf.pages[54].extract_tables()[0]
    tabelas_p56 = pdf.pages[55].extract_tables()

tabela_22 = tabelas_p56[0]
tabela_23 = tabelas_p56[1]

df_21 = pd.DataFrame(tabela_21[1:], columns=["estilo", "volume_litros", "pct_entre_estilos"])
for coluna in ["volume_litros", "pct_entre_estilos"]:
    df_21[coluna] = df_21[coluna].map(converter_numero_br)
df_21["ano_referencia"] = 2025
df_21["fonte"] = "Anuário da Cerveja 2026, Tabela 21, p. 55"
df_21.to_csv(PASTA_SAIDA / "tabela_21_estilos_2025.csv", index=False)

df_22 = pd.DataFrame(tabela_22[1:], columns=["caracteristica", "volume_litros"])
df_22["caracteristica"] = df_22["caracteristica"].str.replace("\n", " ")
df_22["volume_litros"] = df_22["volume_litros"].map(converter_numero_br)
df_22["ano_referencia"] = 2025
df_22["fonte"] = "Anuário da Cerveja 2026, Tabela 22, p. 56"
df_22.to_csv(PASTA_SAIDA / "tabela_22_caracteristicas_2025.csv", index=False)

df_23 = pd.DataFrame(tabela_23[1:], columns=["abrangencia", "volume_litros", "pct_volume", "variacao_pct"])
df_23["abrangencia"] = df_23["abrangencia"].str.replace("\n", " ")
for coluna in ["volume_litros", "pct_volume", "variacao_pct"]:
    df_23[coluna] = df_23[coluna].map(converter_numero_br)
df_23["ano_referencia"] = 2025
df_23["fonte"] = "Anuário da Cerveja 2026, Tabela 23, p. 56"
df_23.to_csv(PASTA_SAIDA / "tabela_23_abrangencia_2025.csv", index=False)

print("Estilos:", len(df_21), "| soma das %:", df_21["pct_entre_estilos"].sum())
print("Faixas de abrangência:", len(df_23), "| soma dos volumes:", df_23["volume_litros"].sum())

display(df_21)
display(df_22)
display(df_23)

Estilos: 17 | soma das %: 100.01
Faixas de abrangência: 5 | soma dos volumes: 15688083191.69


,estilo,volume_litros,pct_entre_estilos,ano_referencia,fonte
0,Malzbier,"40,447,911.89",28.29,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
1,IPA,"29,660,546.34",20.74,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
2,Lager Escura,"22,575,590.79",15.79,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
3,Outras Ales,"16,087,778.35",11.25,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
4,Pale Ale,"12,591,354.44",8.81,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
5,Lagers Claras,"11,587,051.66",8.10,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
6,Ale Belga,"3,280,574.96",2.29,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
7,Weizenbiers,"2,232,771.02",1.56,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
8,Ale Inglesa e Escocesa,"853,484.75",0.60,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"
9,Red Ale,"845,830.70",0.59,2025,"Anuário da Cerveja 2026, Tabela 21, p. 55"


,caracteristica,volume_litros,ano_referencia,fonte
0,Sem Glúten,"367,943,519.39",2025,"Anuário da Cerveja 2026, Tabela 22, p. 56"
1,Gruit,"20,950.00",2025,"Anuário da Cerveja 2026, Tabela 22, p. 56"
2,Com Madeiras Brasileiras*,"56,746.87",2025,"Anuário da Cerveja 2026, Tabela 22, p. 56"
3,Com Fruta,"5,856,369.74",2025,"Anuário da Cerveja 2026, Tabela 22, p. 56"
4,Com Ervas e Especiarias,"4,470,215.70",2025,"Anuário da Cerveja 2026, Tabela 22, p. 56"
5,Com Mel,"1,918,963.63",2025,"Anuário da Cerveja 2026, Tabela 22, p. 56"
6,Com Café,"148,774.49",2025,"Anuário da Cerveja 2026, Tabela 22, p. 56"


,abrangencia,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte
0,Microrregional (raio até 100 km),"28,574,525.31",0.18,-18.34,2025,"Anuário da Cerveja 2026, Tabela 23, p. 56"
1,Regional (raio >100km a 500km),"72,889,348.79",0.46,-12.75,2025,"Anuário da Cerveja 2026, Tabela 23, p. 56"
2,Nacional (>500 km ou no mínimo duas UF),"13,204,006,685.29",84.17,3.04,2025,"Anuário da Cerveja 2026, Tabela 23, p. 56"
3,Nacional e Internacional,"2,354,354,537.90",15.01,-1.04,2025,"Anuário da Cerveja 2026, Tabela 23, p. 56"
4,Somente Internacional,"28,258,094.40",0.18,-12.28,2025,"Anuário da Cerveja 2026, Tabela 23, p. 56"


In [34]:
CAMINHO_PDF_2025 = Path("../data/raw/anuario_cerveja_2025.pdf")

with pdfplumber.open(CAMINHO_PDF_2025) as pdf:
    print("Total de páginas:", len(pdf.pages))

    for numero_tabela in range(16, 24):
        rotulo = f"Tabela {numero_tabela}:"

        for indice, pagina in enumerate(pdf.pages):
            texto = pagina.extract_text() or ""
            if rotulo in texto:
                quantidade = len(pagina.extract_tables())
                print(rotulo, "| índice", indice, "| tabelas detectadas:", quantidade)

Total de páginas: 68
Tabela 16: | índice 47 | tabelas detectadas: 0
Tabela 17: | índice 50 | tabelas detectadas: 0
Tabela 18: | índice 51 | tabelas detectadas: 0
Tabela 19: | índice 52 | tabelas detectadas: 0
Tabela 20: | índice 52 | tabelas detectadas: 0
Tabela 21: | índice 54 | tabelas detectadas: 0
Tabela 22: | índice 55 | tabelas detectadas: 0
Tabela 23: | índice 57 | tabelas detectadas: 0


In [35]:
TOTAL_2024_ORIGINAL = 15_344_065_267.36

regioes = ["Centro-Oeste", "Nordeste", "Norte", "Sudeste", "Sul"]
volumes_2024 = [1_047_137_071.79, 3_290_546_306.35, 221_378_694.16, 8_432_282_117.12, 2_352_721_077.94]

df_regiao_2024 = pd.DataFrame({
    "regiao": regioes,
    "volume_litros": volumes_2024,
    "pct_volume": [6.8, 21.4, 1.4, 55.0, 15.3],
    "variacao_pct": [-14.1, -2.3, -6.3, 2.7, 1.0],
})
df_regiao_2024["ano_referencia"] = 2024
df_regiao_2024["fonte"] = "Anuário da Cerveja 2025, Tabela 17, p. 49 (digitado da imagem)"
df_regiao_2024["observacao"] = ""
df_regiao_2024.loc[df_regiao_2024["regiao"] == "Centro-Oeste", "observacao"] = (
    "Tabela 17 imprime 1.074.137.071,79; usado o valor da Tabela 18 e do texto"
)
df_regiao_2024.to_csv(PASTA_SAIDA / "producao_regiao_2024.csv", index=False)

df_media_2024 = pd.DataFrame({
    "regiao": regioes,
    "volume_litros": volumes_2024,
    "n_cervejarias": [99, 142, 45, 889, 774],
    "producao_media_impressa": [12_307_810.99, 23_729_172.22, 5_252_327.57, 9_231_914.77, 3_010_079.33],
})
df_media_2024["producao_media_calculada"] = (
    df_media_2024["volume_litros"] / df_media_2024["n_cervejarias"]
).round(2)
df_media_2024["ano_referencia"] = 2024
df_media_2024["fonte"] = "Anuário da Cerveja 2025, Tabela 18, p. 50 (digitado da imagem)"
df_media_2024.to_csv(PASTA_SAIDA / "producao_media_regiao_2024.csv", index=False)

diferenca = df_regiao_2024["volume_litros"].sum() - TOTAL_2024_ORIGINAL
print("Diferença entre a soma das regiões e o total:", round(diferenca, 2))
print("Total de cervejarias:", df_media_2024["n_cervejarias"].sum())

display(df_regiao_2024)
display(df_media_2024)

Diferença entre a soma das regiões e o total: -0.0
Total de cervejarias: 1949


,regiao,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte,observacao
0,Centro-Oeste,"1,047,137,071.79",6.80,-14.10,2024,"Anuário da Cerveja 2025, Tabela 17, p. 49 (dig...","Tabela 17 imprime 1.074.137.071,79; usado o va..."
1,Nordeste,"3,290,546,306.35",21.40,-2.30,2024,"Anuário da Cerveja 2025, Tabela 17, p. 49 (dig...",
2,Norte,"221,378,694.16",1.40,-6.30,2024,"Anuário da Cerveja 2025, Tabela 17, p. 49 (dig...",
3,Sudeste,"8,432,282,117.12",55.00,2.70,2024,"Anuário da Cerveja 2025, Tabela 17, p. 49 (dig...",
4,Sul,"2,352,721,077.94",15.30,1.00,2024,"Anuário da Cerveja 2025, Tabela 17, p. 49 (dig...",


,regiao,volume_litros,n_cervejarias,producao_media_impressa,producao_media_calculada,ano_referencia,fonte
0,Centro-Oeste,"1,047,137,071.79",99,"12,307,810.99","10,577,142.14",2024,"Anuário da Cerveja 2025, Tabela 18, p. 50 (dig..."
1,Nordeste,"3,290,546,306.35",142,"23,729,172.22","23,172,861.31",2024,"Anuário da Cerveja 2025, Tabela 18, p. 50 (dig..."
2,Norte,"221,378,694.16",45,"5,252,327.57","4,919,526.54",2024,"Anuário da Cerveja 2025, Tabela 18, p. 50 (dig..."
3,Sudeste,"8,432,282,117.12",889,"9,231,914.77","9,485,131.74",2024,"Anuário da Cerveja 2025, Tabela 18, p. 50 (dig..."
4,Sul,"2,352,721,077.94",774,"3,010,079.33","3,039,691.32",2024,"Anuário da Cerveja 2025, Tabela 18, p. 50 (dig..."


In [36]:
df_conc_2024 = pd.DataFrame({
    "pct_cervejarias": [1.0, 2.0, 3.0, 4.0, 5.0],
    "volume_litros": [7_323_755_354.70, 11_795_322_592.74, 14_261_168_084.14, 15_053_416_371.72, 15_170_103_766.77],
    "pct_volume": [47.73, 76.87, 92.94, 98.11, 98.87],
})
df_conc_2024["ano_referencia"] = 2024
df_conc_2024["fonte"] = "Anuário da Cerveja 2025, Tabela 19, p. 51 (digitado da imagem)"
df_conc_2024["origem"] = "tabela"
df_conc_2024.to_csv(PASTA_SAIDA / "concentracao_2024.csv", index=False)

df_malte_2024 = pd.DataFrame({
    "categoria": ["Cerveja Puro Malte ou 100% malte", "Demais cervejas"],
    "volume_litros": [3_785_009_947.73, 11_559_055_319.63],
    "pct_volume": [24.7, 75.3],
    "variacao_pct": [-15.5, 6.2],
})
df_malte_2024["ano_referencia"] = 2024
df_malte_2024["fonte"] = "Anuário da Cerveja 2025, Tabela 20, p. 51 (digitado da imagem)"
df_malte_2024.to_csv(PASTA_SAIDA / "puro_malte_2024.csv", index=False)

pct_recalculada = (df_conc_2024["volume_litros"] / TOTAL_2024_ORIGINAL * 100).round(2)
print("Porcentagens recalculadas:", pct_recalculada.tolist())
print("Porcentagens impressas:   ", df_conc_2024["pct_volume"].tolist())

diferenca = df_malte_2024["volume_litros"].sum() - TOTAL_2024_ORIGINAL
print("Diferença entre a soma do puro malte e o total:", round(diferenca, 2))

display(df_conc_2024)
display(df_malte_2024)

Porcentagens recalculadas: [47.73, 76.87, 92.94, 98.11, 98.87]
Porcentagens impressas:    [47.73, 76.87, 92.94, 98.11, 98.87]
Diferença entre a soma do puro malte e o total: -0.0


,pct_cervejarias,volume_litros,pct_volume,ano_referencia,fonte,origem
0,1.00,"7,323,755,354.70",47.73,2024,"Anuário da Cerveja 2025, Tabela 19, p. 51 (dig...",tabela
1,2.00,"11,795,322,592.74",76.87,2024,"Anuário da Cerveja 2025, Tabela 19, p. 51 (dig...",tabela
2,3.00,"14,261,168,084.14",92.94,2024,"Anuário da Cerveja 2025, Tabela 19, p. 51 (dig...",tabela
3,4.00,"15,053,416,371.72",98.11,2024,"Anuário da Cerveja 2025, Tabela 19, p. 51 (dig...",tabela
4,5.00,"15,170,103,766.77",98.87,2024,"Anuário da Cerveja 2025, Tabela 19, p. 51 (dig...",tabela


,categoria,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte
0,Cerveja Puro Malte ou 100% malte,"3,785,009,947.73",24.70,-15.50,2024,"Anuário da Cerveja 2025, Tabela 20, p. 51 (dig..."
1,Demais cervejas,"11,559,055,319.63",75.30,6.20,2024,"Anuário da Cerveja 2025, Tabela 20, p. 51 (dig..."


In [37]:
estilos_2024 = [
    ("Ale Alemã", 269_492.80, -66.9),
    ("Ale Americana Intensa", 907_252.22, 43.45),
    ("Ale Americana Leve", 2_106_770.47, -18.35),
    ("Ale Belga Intensa", 434_923.41, -9.16),
    ("Ale Belga Leve", 2_998_513.63, 63.46),
    ("Ale Inglesa Intensa", 73_890.38, -54.07),
    ("Ale Inglesa Moderada", 270_445.60, 86.19),
    ("Bitter Inglesa", 164_530.43, 35.5),
    ("Catharina Sour", 410_534.78, 54.46),
    ("Cerveja Gruit", 400.00, float("nan")),
    ("Cerveja Selvagem Brasileira", 20_038.28, -67.13),
    ("IPA", 29_913_104.80, -9.54),
    ("Lager Intensa Escura", 1_137_989.35, 105.25),
    ("Lager Leve Clara", 8_950_879_380.11, 13.18),
    ("Lager Leve Escura", 17_547_355.74, -26.76),
    ("Malzbier", 46_701_564.39, 12.91),
    ("Pale Ale", 2_937_193.14, -33.02),
    ("Pilsener", 4_974_576_556.47, 39.76),
    ("Porter", 324_243.20, 79.61),
    ("Red Ale", 1_199_818.53, -7.31),
    ("Saison", 202_045.28, 10.22),
    ("Scottish Ale", 25_375.14, 68.84),
    ("Stout", 827_404.39, 17.54),
    ("Weizenbier", 2_736_030.90, -71.42),
    ("Outras Ales", 8_743_739.48, -39.31),
    ("Outras Lagers", 1_296_805_239.13, -65.43),
    ("Outras", 1_851_435.31, -61.88),
]

df_estilos_2024 = pd.DataFrame(estilos_2024, columns=["estilo", "volume_litros", "variacao_pct"])
df_estilos_2024["ano_referencia"] = 2024
df_estilos_2024["fonte"] = "Anuário da Cerveja 2025, Tabela 22, p. 54 (digitado da imagem)"
df_estilos_2024.to_csv(PASTA_SAIDA / "estilos_2024.csv", index=False)

diferenca = df_estilos_2024["volume_litros"].sum() - TOTAL_2024_ORIGINAL
print("Estilos:", len(df_estilos_2024))
print("Diferença entre a soma dos estilos e o total:", round(diferenca, 2))
df_estilos_2024

Estilos: 27
Diferença entre a soma dos estilos e o total: -0.0


,estilo,volume_litros,variacao_pct,ano_referencia,fonte
0,Ale Alemã,"269,492.80",-66.90,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
1,Ale Americana Intensa,"907,252.22",43.45,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
2,Ale Americana Leve,"2,106,770.47",-18.35,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
3,Ale Belga Intensa,"434,923.41",-9.16,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
4,Ale Belga Leve,"2,998,513.63",63.46,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
5,Ale Inglesa Intensa,"73,890.38",-54.07,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
6,Ale Inglesa Moderada,"270,445.60",86.19,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
7,Bitter Inglesa,"164,530.43",35.50,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
8,Catharina Sour,"410,534.78",54.46,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."
9,Cerveja Gruit,400.00,NaN,2024,"Anuário da Cerveja 2025, Tabela 22, p. 54 (dig..."


In [38]:
caracteristicas_2024 = [
    ("Sem glúten", 71_075_867.37, 40_388_707.07, 131.6),
    ("Com fruta", 5_854_467.02, -3_539_434.35, -37.7),
    ("Com madeira*", 625_713_550.62, 219_798_233.06, 54.1),
    ("Com mel", 3_302_588.45, 1_266_133.55, 62.2),
    ("Com chocolate", 109_449.52, 15_341.54, 16.3),
    ("Com café", 306_160.27, 179_074.28, 140.9),
    ("Com pimenta", 114_286.58, 20_172.83, 21.4),
    ("Com ervas e especiarias", 8_498_677.98, -1_527_974.61, -15.2),
    ("Com abóbora", 22_340.37, -14_996.63, -40.2),
    ("Com leite e derivados", 32_137.30, -2_118.30, -6.2),
    ("Defumada", 54_703.75, -1_146.50, -2.1),
]

df_carac_2024 = pd.DataFrame(
    caracteristicas_2024,
    columns=["caracteristica", "volume_litros", "variacao_absoluta_litros", "variacao_pct"],
)
df_carac_2024["ano_referencia"] = 2024
df_carac_2024["fonte"] = "Anuário da Cerveja 2025, Tabela 23, p. 56 (digitado da imagem)"
df_carac_2024.to_csv(PASTA_SAIDA / "caracteristicas_2024.csv", index=False)

volume_2023 = df_carac_2024["volume_litros"] - df_carac_2024["variacao_absoluta_litros"]
df_carac_2024["variacao_recalculada"] = (df_carac_2024["variacao_absoluta_litros"] / volume_2023 * 100).round(1)

df_abrang_2024 = pd.DataFrame({
    "abrangencia": [
        "Microrregional (raio até 100 km)",
        "Regional (raio >100km a 500km)",
        "Nacional (>500 km ou no mínimo duas UF)",
        "Nacional e Internacional",
        "Somente Internacional",
    ],
    "volume_litros": [34_992_867.48, 83_539_433.00, 12_814_184_421.93, 2_379_134_631.41, 32_213_913.54],
    "pct_volume": [0.23, 0.54, 83.51, 15.51, 0.21],
    "variacao_pct": [4.9, 5.2, -3.0, 18.1, 39.7],
})
df_abrang_2024["ano_referencia"] = 2024
df_abrang_2024["fonte"] = "Anuário da Cerveja 2025, Tabela 24, p. 57 (digitado da imagem)"
df_abrang_2024.to_csv(PASTA_SAIDA / "abrangencia_2024.csv", index=False)

diferenca = df_abrang_2024["volume_litros"].sum() - TOTAL_2024_ORIGINAL
print("Diferença entre a soma da abrangência e o total:", round(diferenca, 2))

display(df_carac_2024)
display(df_abrang_2024)

Diferença entre a soma da abrangência e o total: 0.0


,caracteristica,volume_litros,variacao_absoluta_litros,variacao_pct,ano_referencia,fonte,variacao_recalculada
0,Sem glúten,"71,075,867.37","40,388,707.07",131.60,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",131.60
1,Com fruta,"5,854,467.02","-3,539,434.35",-37.70,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",-37.70
2,Com madeira*,"625,713,550.62","219,798,233.06",54.10,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",54.10
3,Com mel,"3,302,588.45","1,266,133.55",62.20,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",62.20
4,Com chocolate,"109,449.52","15,341.54",16.30,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",16.30
5,Com café,"306,160.27","179,074.28",140.90,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",140.90
6,Com pimenta,"114,286.58","20,172.83",21.40,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",21.40
7,Com ervas e especiarias,"8,498,677.98","-1,527,974.61",-15.20,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",-15.20
8,Com abóbora,"22,340.37","-14,996.63",-40.20,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",-40.20
9,Com leite e derivados,"32,137.30","-2,118.30",-6.20,2024,"Anuário da Cerveja 2025, Tabela 23, p. 56 (dig...",-6.20


,abrangencia,volume_litros,pct_volume,variacao_pct,ano_referencia,fonte
0,Microrregional (raio até 100 km),"34,992,867.48",0.23,4.90,2024,"Anuário da Cerveja 2025, Tabela 24, p. 57 (dig..."
1,Regional (raio >100km a 500km),"83,539,433.00",0.54,5.20,2024,"Anuário da Cerveja 2025, Tabela 24, p. 57 (dig..."
2,Nacional (>500 km ou no mínimo duas UF),"12,814,184,421.93",83.51,-3.00,2024,"Anuário da Cerveja 2025, Tabela 24, p. 57 (dig..."
3,Nacional e Internacional,"2,379,134,631.41",15.51,18.10,2024,"Anuário da Cerveja 2025, Tabela 24, p. 57 (dig..."
4,Somente Internacional,"32,213,913.54",0.21,39.70,2024,"Anuário da Cerveja 2025, Tabela 24, p. 57 (dig..."


In [39]:
cervejarias_uf_2025 = [
    ("AP", "Norte", 1, 0.0),
    ("RR", "Norte", 1, 0.0),
    ("AC", "Norte", 2, 0.0),
    ("AM", "Norte", 4, -20.0),
    ("TO", "Norte", 4, -33.3),
    ("PI", "Nordeste", 5, -16.7),
    ("SE", "Nordeste", 5, 0.0),
    ("AL", "Nordeste", 7, -12.5),
    ("RO", "Norte", 10, 25.0),
    ("PB", "Nordeste", 11, 57.1),
    ("MS", "Centro-Oeste", 12, 9.1),
    ("MA", "Nordeste", 15, 7.1),
    ("RN", "Nordeste", 16, -40.7),
    ("PE", "Nordeste", 17, -15.0),
    ("DF", "Centro-Oeste", 19, 0.0),
    ("PA", "Norte", 22, 0.0),
    ("CE", "Nordeste", 25, 8.7),
    ("BA", "Nordeste", 29, -9.4),
    ("MT", "Centro-Oeste", 29, -3.3),
    ("GO", "Centro-Oeste", 38, -2.6),
    ("ES", "Sudeste", 97, 7.8),
    ("RJ", "Sudeste", 138, 6.2),
    ("PR", "Sul", 187, 6.9),
    ("MG", "Sudeste", 236, -2.5),
    ("SC", "Sul", 247, -1.2),
    ("RS", "Sul", 325, -6.9),
    ("SP", "Sudeste", 452, 5.9),
]

df_cerv_uf = pd.DataFrame(cervejarias_uf_2025, columns=["uf", "regiao", "n_cervejarias", "variacao_pct"])
df_cerv_uf["ano_referencia"] = 2025
df_cerv_uf["fonte"] = "Anuário da Cerveja 2026, Gráfico 2 (digitado da imagem)"
df_cerv_uf["observacao"] = "Legenda do gráfico diz 2023/2024; a variação é 2024/2025"
df_cerv_uf.to_csv(PASTA_SAIDA / "grafico_02_cervejarias_uf_2025.csv", index=False)

print("UFs:", len(df_cerv_uf))
print("Total de cervejarias:", df_cerv_uf["n_cervejarias"].sum())

soma_por_regiao = df_cerv_uf.groupby("regiao")["n_cervejarias"].sum()
display(soma_por_regiao)
display(df_18[["regiao", "n_cervejarias"]].sort_values("regiao"))

UFs: 27
Total de cervejarias: 1954


regiao
Centro-Oeste     98
Nordeste        130
Norte            44
Sudeste         923
Sul             759
Name: n_cervejarias, dtype: int64

,regiao,n_cervejarias
1,Centro-Oeste,98
0,Nordeste,130
2,Norte,44
4,Sudeste,923
3,Sul,759


In [40]:
DATA_CONSULTA_COMEX = "2026-10-09"
PASTA_BRUTOS = Path("../data/raw")

def ler_comex(caminho, fluxo):
    df = pd.read_csv(caminho, sep=";", encoding="utf-8-sig", dtype=str)
    df.columns = ["ano", "mes_texto", "sh4", "descricao", "valor_fob_usd", "kg_liquido"]

    for coluna in df.columns:
        df[coluna] = df[coluna].str.strip()

    df["ano"] = df["ano"].astype(int)
    df["mes"] = df["mes_texto"].str[:2].astype(int)
    df["valor_fob_usd"] = df["valor_fob_usd"].astype(int)
    df["kg_liquido"] = df["kg_liquido"].astype(int)
    df["data"] = pd.to_datetime({"year": df["ano"], "month": df["mes"], "day": 1})
    df["fluxo"] = fluxo

    colunas_finais = ["data", "ano", "mes", "fluxo", "sh4", "descricao", "valor_fob_usd", "kg_liquido"]
    return df[colunas_finais]

df_imp = ler_comex(PASTA_BRUTOS / "comex_importacao_2015_2025.csv", "importacao")
df_exp = ler_comex(PASTA_BRUTOS / "comex_exportacao_2015_2025.csv", "exportacao")

df_comex = pd.concat([df_imp, df_exp], ignore_index=True)
df_comex = df_comex.sort_values(["fluxo", "sh4", "data"]).reset_index(drop=True)
df_comex["fonte"] = "Comex Stat (MDIC), consulta em " + DATA_CONSULTA_COMEX

df_insumos = df_comex[df_comex["sh4"].isin(["1107", "1210"])]
df_insumos.to_csv(PASTA_SAIDA / "comex_insumos_mensal_2015_2025.csv", index=False)

print("Linhas:", len(df_comex))
print("Período:", df_comex["data"].min().date(), "a", df_comex["data"].max().date())
display(df_comex.groupby(["fluxo", "sh4"]).size().unstack())

cerveja_2024 = df_comex[(df_comex["sh4"] == "2203") & (df_comex["ano"] == 2024)]
display(cerveja_2024.groupby("fluxo")[["valor_fob_usd", "kg_liquido"]].sum())

Linhas: 699
Período: 2015-01-01 a 2025-12-01


sh4,1107,1210,2203
fluxo,,,
exportacao,107,64,132
importacao,132,132,132


,valor_fob_usd,kg_liquido
fluxo,,
exportacao,201168175,291501225
importacao,8281505,7836296


In [41]:
def ler_comex_ncm(caminho, fluxo):
    df = pd.read_csv(caminho, sep=";", encoding="utf-8-sig", dtype=str)
    df.columns = ["ano", "mes_texto", "ncm", "descricao", "unidade", "valor_fob_usd", "kg_liquido", "litros"]

    for coluna in df.columns:
        df[coluna] = df[coluna].str.strip()

    print(fluxo, "| unidades encontradas:", df["unidade"].unique())

    df["ano"] = df["ano"].astype(int)
    df["mes"] = df["mes_texto"].str[:2].astype(int)
    for coluna in ["valor_fob_usd", "kg_liquido", "litros"]:
        df[coluna] = df[coluna].astype(int)
    df["data"] = pd.to_datetime({"year": df["ano"], "month": df["mes"], "day": 1})
    df["fluxo"] = fluxo

    colunas_finais = ["data", "ano", "mes", "fluxo", "ncm", "descricao", "valor_fob_usd", "kg_liquido", "litros"]
    return df[colunas_finais]

df_cerveja_imp = ler_comex_ncm(PASTA_BRUTOS / "comex_cerveja_ncm_importacao_2015_2025.csv", "importacao")
df_cerveja_exp = ler_comex_ncm(PASTA_BRUTOS / "comex_cerveja_ncm_exportacao_2015_2025.csv", "exportacao")

df_cerveja = pd.concat([df_cerveja_imp, df_cerveja_exp], ignore_index=True)
df_cerveja = df_cerveja.sort_values(["fluxo", "ncm", "data"]).reset_index(drop=True)
df_cerveja["fonte"] = "Comex Stat (MDIC), consulta em " + DATA_CONSULTA_COMEX
df_cerveja.to_csv(PASTA_SAIDA / "comex_cerveja_mensal_2015_2025.csv", index=False)

display(df_cerveja.groupby(["fluxo", "ncm"]).size().unstack())
display(df_cerveja.groupby(["fluxo", "ncm"])["data"].min().unstack())

cerveja_2024 = df_cerveja[df_cerveja["ano"] == 2024]
display(cerveja_2024.groupby("fluxo")[["valor_fob_usd", "litros"]].sum())

importacao | unidades encontradas: <ArrowStringArray>
['LITRO']
Length: 1, dtype: str
exportacao | unidades encontradas: <ArrowStringArray>
['LITRO']
Length: 1, dtype: str


ncm,22029100,22030000
fluxo,,
exportacao,105,132
importacao,107,132


ncm,22029100,22030000
fluxo,,
exportacao,2017-02-01,2015-01-01
importacao,2017-01-01,2015-01-01


,valor_fob_usd,litros
fluxo,,
exportacao,204290418,332540550
importacao,9259422,7493971


diagnóstico arquivo CAGED

In [42]:
df_caged_bruto = pd.read_csv(PASTA_BRUTOS / "caged_cerveja_2020_2025.csv")

print("Linhas:", len(df_caged_bruto))
print("Meses distintos:", len(df_caged_bruto[["ano", "mes"]].drop_duplicates()))
display(df_caged_bruto.groupby("ano")["mes"].nunique())

confere = (df_caged_bruto["admissoes"] - df_caged_bruto["desligamentos"] == df_caged_bruto["saldo"]).all()
print("Admissões - desligamentos = saldo em todas as linhas:", confere)

display(df_caged_bruto.groupby("indicador_fora_prazo")[["admissoes", "desligamentos", "saldo"]].sum())

Linhas: 1745
Meses distintos: 72


ano
2020    12
2021    12
2022    12
2023    12
2024    12
2025    12
Name: mes, dtype: int64

Admissões - desligamentos = saldo em todas as linhas: True


,admissoes,desligamentos,saldo
indicador_fora_prazo,,,
0,57111,57389,-278


In [43]:
df_caged = (
    df_caged_bruto
    .groupby(["ano", "mes", "sigla_uf"], as_index=False)[["admissoes", "desligamentos", "saldo"]]
    .sum()
    .rename(columns={"sigla_uf": "uf"})
)
df_caged["data"] = pd.to_datetime({"year": df_caged["ano"], "month": df_caged["mes"], "day": 1})
df_caged["fonte"] = "Novo CAGED via Base dos Dados (BigQuery), CNAE 1113-5/02; consulta em src/consulta_caged.sql"
df_caged.to_csv(PASTA_SAIDA / "caged_cerveja_mensal_2020_2025.csv", index=False)

saldo_2025 = df_caged[df_caged["ano"] == 2025].groupby("uf")["saldo"].sum()

t16 = pd.read_csv(PASTA_SAIDA / "tabela_16_empregos_uf_2025.csv")
t16["variacao_implicita"] = (t16["estoque_empregos"] * t16["variacao_pct"] / (100 + t16["variacao_pct"])).round()

comparacao = (
    t16.set_index("uf")[["estoque_empregos", "variacao_pct", "variacao_implicita"]]
    .join(saldo_2025.rename("saldo_caged_2025"))
)
display(comparacao.sort_values("estoque_empregos", ascending=False).head(10))

,estoque_empregos,variacao_pct,variacao_implicita,saldo_caged_2025
uf,,,,
SP,14182,-2.82,-412.00,-418
RJ,5216,-3.19,-172.00,-164
MG,4720,3.74,170.00,175
PR,2680,-5.50,-156.00,-162
RS,2479,-1.70,-43.00,-33
PE,2288,-3.74,-89.00,-89
BA,2234,-0.18,-4.00,-2
SC,1552,-8.81,-150.00,-152
GO,1535,-1.85,-29.00,-29
